# Livrable 1 — Modélisation formelle du problème

**Projet :** Optimisation de tournées de livraison — Mobilité Multimodale Intelligente  
**Commanditaire :** ADEME (Agence de l'Environnement et de la Maîtrise de l'Énergie)  
**Structure :** CesiCDP  
**Date :** Avril 2026

---

## Résumé

Ce notebook présente la modélisation formelle du problème d'optimisation de tournées de livraison dans un réseau routier. Nous reformulons le problème de manière rigoureuse, en y intégrant deux contraintes supplémentaires choisies :

1. **Fenêtres temporelles** (*Time Windows*) : chaque ville ne peut être visitée que dans un intervalle de temps défini.  
2. **Routes dynamiques et perturbations** : les coûts de transit entre villes peuvent varier en cours de tournée (travaux, accidents, conditions météo).

Les méthodes de résolution ne sont pas abordées dans ce livrable.

---

## 1. Contexte et motivation

La transition énergétique impose de repenser en profondeur la logistique du transport. La réduction des émissions de gaz à effet de serre passe notamment par l'optimisation des tournées de livraison : moins de kilomètres parcourus signifie moins de carburant consommé et moins de CO₂ émis.

Dans ce cadre, l'ADEME finance des projets de **Mobilité Multimodale Intelligente**. Notre mission au sein de CesiCDP est de proposer un modèle algorithmique capable de calculer, sur un réseau routier, une tournée optimisée permettant :

- de relier un sous-ensemble de villes (points de livraison),
- de revenir au dépôt de départ,
- en **minimisant la durée totale** du parcours,
- tout en **respectant des contraintes réelles** : plages horaires de livraison et aléas routiers.

Ce problème s'inscrit dans la famille des **problèmes de tournée de véhicules** (*Vehicle Routing Problems*, VRP), dont le cas le plus fondamental est le **problème du voyageur de commerce** (*Travelling Salesman Problem*, TSP).

---

## 2. Modélisation formelle

### 2.1 Représentation du réseau routier

Le réseau routier est modélisé par un **graphe non orienté pondéré** :

$$G = (V, E)$$

où :

- $V = \{v_0, v_1, \dots, v_n\}$ est l'ensemble des **sommets** (nœuds), avec $|V| = n + 1$ :
  - $v_0$ : le **dépôt** (point de départ et d'arrivée du véhicule),
  - $v_1, \dots, v_n$ : les $n$ **clients** (points de livraison).

- $E \subseteq \binom{V}{2}$ est l'ensemble des **arêtes** (routes bidirectionnelles entre deux villes).

Chaque arête $\{i, j\} \in E$ est associée à un **coût de transit** $c_{ij}(t)$, qui représente le temps de trajet entre $v_i$ et $v_j$ **en partant à l'instant $t$** (voir section 2.3 sur les perturbations dynamiques).

Les coûts sont **symétriques** : $c_{ij}(t) = c_{ji}(t)$ pour tout $t$, ce qui reflète le fait qu'une route peut être empruntée dans les deux sens avec le même temps de trajet.

> **Note :** dans notre modèle, le graphe est supposé **complet** (tout couple de villes est relié par une arête), ce qui est justifié dans le cadre d'un réseau routier où il est toujours possible de rejoindre une ville, directement ou via un chemin indirect.

### 2.2 Contrainte 1 — Fenêtres temporelles (*Time Windows*)

Chaque client $v_i \in V \setminus \{v_0\}$ est associé à une **fenêtre temporelle** :

$$[a_i,\ b_i], \quad a_i \leq b_i$$

où :
- $a_i$ : heure d'ouverture (plus tôt possible pour débuter la livraison),
- $b_i$ : heure de fermeture (deadline, au-delà de laquelle la livraison est impossible).

Soit $\tau_i$ l'heure d'**arrivée** du véhicule en $v_i$. La contrainte de fenêtre temporelle impose :

$$a_i \leq \tau_i \leq b_i, \quad \forall i \in \{1, \dots, n\}$$

**Attente autorisée :** si le véhicule arrive avant l'ouverture ($\tau_i < a_i$), il peut attendre. L'heure de **départ** de $v_i$ est :

$$d_i = \max(\tau_i,\ a_i) + s_i$$

où $s_i \geq 0$ est le **temps de service** (temps nécessaire pour effectuer la livraison en $v_i$).

**Relation de récurrence temporelle :** si le véhicule visite les clients dans l'ordre $\sigma = (\sigma_0, \sigma_1, \dots, \sigma_n, \sigma_0)$, avec $\sigma_0 = 0$ (dépôt), alors :

$$\tau_{\sigma_{k+1}} = d_{\sigma_k} + c_{\sigma_k, \sigma_{k+1}}\!\left(d_{\sigma_k}\right)$$

Le dépôt $v_0$ possède aussi une fenêtre $[a_0, b_0]$ représentant la plage d'activité du dépôt (par exemple, la journée de travail).

### 2.3 Contrainte 2 — Routes dynamiques et perturbations

Dans un réseau routier réel, les temps de trajet ne sont pas fixes. Des **perturbations** peuvent survenir en cours de tournée : accidents, travaux imprévus, conditions météorologiques, congestions.

Nous modélisons les coûts de transit comme des **fonctions dépendantes du temps** :

$$c_{ij} : \mathbb{R}_{\geq 0} \to \mathbb{R}_{> 0}$$

$$c_{ij}(t) = c_{ij}^{\text{base}} \cdot \left(1 + \delta_{ij}(t)\right)$$

où :
- $c_{ij}^{\text{base}} > 0$ : coût nominal (temps de trajet en conditions normales),
- $\delta_{ij}(t) \geq -1$ : **facteur de perturbation** à l'instant $t$, avec $\delta_{ij}(t) > 0$ signifiant un ralentissement et $-1 < \delta_{ij}(t) < 0$ une accélération (par exemple, voie express ouverte).

**Modèle de perturbation par événements discrets :** une perturbation est un triplet $\{i, j\}, [t_{\text{début}}, t_{\text{fin}}], \alpha)$ qui applique un facteur multiplicatif $\alpha > 0$ sur l'arête $\{i, j\}$ pendant l'intervalle $[t_{\text{début}}, t_{\text{fin}}]$ — dans les deux sens de parcours :

$$\delta_{ij}(t) = \delta_{ji}(t) = \begin{cases} \alpha - 1 & \text{si } t \in [t_{\text{début}},\ t_{\text{fin}}] \\ 0 & \text{sinon} \end{cases}$$

L'ensemble des perturbations actives à l'instant $t$ est noté $\mathcal{P}(t)$.

> **Hypothèse de causalité :** les perturbations sont **connues à l'avance** dans notre modèle (perturbations planifiées ou prévisibles). L'extension à des perturbations stochastiques constitue une piste d'amélioration présentée en section 6.

### 2.4 Définition formelle du problème (TSPTW-D)

Nous nommons notre problème **TSPTW-D** (*Travelling Salesman Problem with Time Windows and Dynamic disruptions*).

**Données d'entrée :**

| Symbole | Description |
|---|---|
| $G = (V, E)$ | Graphe non orienté complet, $\|V\| = n+1$ |
| $c_{ij}^{\text{base}} = c_{ji}^{\text{base}}$ | Coût nominal symétrique de l'arête $\{i,j\}$ |
| $[a_i, b_i]$ | Fenêtre temporelle du client $v_i$ |
| $s_i$ | Temps de service au client $v_i$ |
| $\mathcal{P}$ | Ensemble des perturbations $(\{i, j\}, t_{\text{début}}, t_{\text{fin}}, \alpha)$ |

**Variable de décision :**

$$\sigma = (\sigma_0, \sigma_1, \dots, \sigma_n, \sigma_0), \quad \sigma_0 = 0$$

une permutation des clients $\{1, \dots, n\}$ représentant l'ordre de visite.

**Fonction objectif — minimiser la durée totale de la tournée :**

$$\min_{\sigma} \; Z(\sigma) = \tau_{\sigma_0}^{\text{retour}} - d_{\sigma_0}^{\text{départ}}$$

ce qui revient à minimiser le temps de retour au dépôt :

$$\min_{\sigma} \; \tau_0^{\text{retour}}$$

**Contraintes :**

$$\begin{cases}
a_i \leq \tau_{\sigma_k} \leq b_i & \forall k \in \{1,\dots,n\},\ i = \sigma_k \quad \text{(fenêtres temporelles)} \\
\tau_{\sigma_{k+1}} = \max(\tau_{\sigma_k}, a_{\sigma_k}) + s_{\sigma_k} + c_{\sigma_k, \sigma_{k+1}}\!\left(d_{\sigma_k}\right) & \forall k \in \{0,\dots,n-1\} \quad \text{(propagation temporelle)} \\
c_{ij}(t) = c_{ji}(t) = c_{ij}^{\text{base}} \cdot (1 + \delta_{ij}(t)) & \forall \{i,j\} \in E,\ t \geq 0 \quad \text{(coûts dynamiques symétriques)} \\
\sigma \text{ est une permutation de } \{0,\dots,n\} & \quad \text{(chaque ville visitée exactement une fois)}
\end{cases}$$

---

## 3. Illustration du modèle

La cellule de code suivante génère une visualisation d'un exemple d'instance TSPTW-D : un réseau de 6 nœuds avec fenêtres temporelles et une perturbation active sur une arête.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# ----- Instance exemple -----
np.random.seed(42)
n_clients = 5  # + 1 dépôt = 6 nœuds
n_nodes = n_clients + 1

# Coordonnées géographiques (pour la visualisation)
coords = {
    0: (0.5, 0.5),   # dépôt
    1: (0.1, 0.8),
    2: (0.9, 0.85),
    3: (0.75, 0.2),
    4: (0.2, 0.15),
    5: (0.55, 0.95),
}

# Fenêtres temporelles [a_i, b_i] et temps de service s_i (en minutes)
time_windows = {
    0: (0,   480),   # dépôt : journée complète (8h)
    1: (30,  120),
    2: (60,  180),
    3: (90,  200),
    4: (150, 300),
    5: (20,   90),
}
service_times = {i: 10 for i in range(n_nodes)}

# Coûts de base (temps en minutes, calculés depuis les coordonnées)
def euclidean_minutes(c1, c2, scale=200):
    return round(np.hypot(c1[0]-c2[0], c1[1]-c2[1]) * scale, 1)

cost_base = {}
for i in range(n_nodes):
    for j in range(n_nodes):
        if i != j:
            cost_base[(i, j)] = euclidean_minutes(coords[i], coords[j])

# Perturbations : arc (2,3) perturbé de t=80 à t=160 (facteur x2.5)
perturbations = [
    {"arc": (2, 3), "t_start": 80, "t_end": 160, "alpha": 2.5},
]

def c_ij(i, j, t, cost_base, perturbations):
    """Coût dynamique de l'arc (i,j) au temps t."""
    base = cost_base[(i, j)]
    for p in perturbations:
        if p["arc"] == (i, j) and p["t_start"] <= t <= p["t_end"]:
            return base * p["alpha"]
    return base

# ----- Visualisation -----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Graphe du réseau ---
ax = axes[0]
G = nx.Graph()
G.add_nodes_from(range(n_nodes))
# Ajout d'un sous-ensemble d'arcs pour la lisibilité
visible_arcs = [(i, j) for i in range(n_nodes) for j in range(n_nodes) if i != j]
edge_colors = []
edge_widths = []
for (i, j) in visible_arcs:
    is_perturbed = any(p["arc"] == (i, j) for p in perturbations)
    edge_colors.append("red" if is_perturbed else "#aaaaaa")
    edge_widths.append(3.0 if is_perturbed else 0.8)
    G.add_edge(i, j)

node_colors = ["#f4a261" if i == 0 else "#457b9d" for i in range(n_nodes)]
labels = {i: f"v{i}\n[{time_windows[i][0]},{time_windows[i][1]}]" for i in range(n_nodes)}
labels[0] = f"Dépôt\n[{time_windows[0][0]},{time_windows[0][1]}]"

nx.draw_networkx(G, pos=coords, ax=ax, labels=labels,
                 node_color=node_colors, node_size=1200,
                 edge_color=edge_colors, width=edge_widths,
                 font_size=7)

depot_patch = mpatches.Patch(color="#f4a261", label="Dépôt (v₀)")
client_patch = mpatches.Patch(color="#457b9d", label="Client vᵢ")
perturb_patch = mpatches.Patch(color="red", label="Arête perturbée (2—3)")
ax.legend(handles=[depot_patch, client_patch, perturb_patch], loc="lower right", fontsize=8)
ax.set_title("Réseau routier — Instance TSPTW-D\n(valeurs [aᵢ, bᵢ] en minutes)", fontsize=11)
ax.axis("off")

# --- Coût dynamique de l'arc perturbé ---
ax2 = axes[1]
t_range = np.linspace(0, 300, 500)
arc = (2, 3)
costs = [c_ij(*arc, t, cost_base, perturbations) for t in t_range]
base_line = [cost_base[arc]] * len(t_range)

ax2.plot(t_range, costs, color="red", linewidth=2, label=f"c₂₃(t) dynamique")
ax2.plot(t_range, base_line, color="gray", linewidth=1.5, linestyle="--", label=f"c₂₃ nominal = {cost_base[arc]} min")
ax2.axvspan(80, 160, alpha=0.15, color="red", label="Perturbation active (α=2.5)")
ax2.set_xlabel("Temps t (minutes)", fontsize=10)
ax2.set_ylabel("Coût de transit (minutes)", fontsize=10)
ax2.set_title("Coût dynamique de l'arc (v₂ → v₃)", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("tsptwd_illustration.png", dpi=120, bbox_inches="tight")
plt.show()
print("Instance générée avec succès.")


---

## 4. Analyse de la complexité

### 4.1 TSP classique : NP-difficulté

Le problème du voyageur de commerce (TSP) sans contrainte est l'un des problèmes d'optimisation combinatoire les plus étudiés. Sa complexité est établie depuis les travaux fondateurs de Karp (1972) [1].

**Taille de l'espace de recherche :** Pour $n$ clients, le nombre de tournées candidates est $(n-1)!/2$ (en version symétrique). Pour $n = 20$, cela représente plus de $6 \times 10^{16}$ tournées — une énumération exhaustive est impossible en pratique.

**Théorème (Karp, 1972) :** *Le TSP de décision — "existe-t-il une tournée de coût $\leq K$ ?" — est NP-complet.*

*Preuve (réduction depuis le problème du cycle hamiltonien, lui-même NP-complet) :*

Soit un graphe $H = (V, E)$ quelconque. On construit une instance TSP avec $c_{ij} = 1$ si $(i,j) \in E$, et $c_{ij} = 2$ sinon. Il existe un cycle hamiltonien dans $H$ si et seulement si le TSP admet une solution de coût $\leq n$. La réduction est polynomiale. $\square$

### 4.2 TSPTW : NP-difficulté préservée

L'ajout de fenêtres temporelles ne simplifie pas le problème. Savelsbergh (1985) [2] a montré que **vérifier la faisabilité d'une instance TSPTW est déjà NP-complet**.

**Lemme :** *Le TSPTW est NP-difficile au sens fort.*

*Idée de la preuve :* On peut encoder le TSP dans le TSPTW en posant $a_i = 0$ et $b_i = +\infty$ pour tous les nœuds (fenêtres non contraignantes). Les fenêtres temporelles restrictives permettent en outre de modéliser des contraintes d'ordonnancement, ce qui étend la difficulté.

Les fenêtres temporelles peuvent toutefois **réduire l'espace de recherche effectif** en éliminant rapidement les permutations infaisables, ce qui est exploité par les algorithmes de résolution (branchement et élagage).

### 4.3 TSPTW-D : complexité en présence de dynamisme

L'ajout de perturbations dynamiques (coûts dépendant du temps) ne change pas la classe de complexité : le TSPTW-D reste **NP-difficile**, puisque le TSPTW s'y réduit trivialement (en posant $\delta_{ij}(t) = 0$ pour tout arc et tout instant).

Cependant, la dynamicité introduit une **complexité algorithmique supplémentaire** :

- Le calcul de $c_{ij}(t)$ n'est plus $O(1)$ : il dépend du nombre de perturbations actives $|\mathcal{P}|$.
- La propagation temporelle $\tau_{\sigma_{k+1}} = f(\tau_{\sigma_k}, \sigma_k, \sigma_{k+1})$ devient **non linéaire** et **non stationnaire**, ce qui invalide les optimisations basées sur la sous-additivité des coûts.
- La **propriété FIFO** (First-In-First-Out) — partir plus tôt ne peut pas faire arriver plus tard — est supposée vérifiée pour que les algorithmes de plus court chemin classiques restent applicables sur le graphe temporel.

**Tableau récapitulatif de complexité :**

| Problème | Complexité | Référence |
|---|---|---|
| TSP (décision) | NP-complet | Karp, 1972 [1] |
| TSP (optimisation) | NP-difficile | — |
| TSPTW (faisabilité) | NP-complet | Savelsbergh, 1985 [2] |
| TSPTW (optimisation) | NP-difficile | — |
| TSPTW-D (optimisation) | NP-difficile | (par réduction depuis TSPTW) |

In [ ]:
import math
import matplotlib.pyplot as plt

# Illustration de la croissance factorielle de l'espace de recherche
ns = list(range(2, 21))
search_space = [math.factorial(n - 1) // 2 for n in ns]  # (n-1)!/2 pour TSP symétrique

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(ns, search_space, marker="o", color="#e76f51", linewidth=2)
ax.set_xlabel("Nombre de clients n", fontsize=11)
ax.set_ylabel("Nombre de tournées candidates (échelle log)", fontsize=11)
ax.set_title("Explosion combinatoire : taille de l'espace de recherche TSP = (n−1)!/2", fontsize=11)
ax.grid(True, which="both", alpha=0.3)

# Annotation pour n=20
ax.annotate(f"n=20 : {search_space[-1]:.2e} tournées",
            xy=(20, search_space[-1]),
            xytext=(14, search_space[-1] * 0.01),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=9)

plt.tight_layout()
plt.show()


---

## 5. Comparaison TSP — TSPTW — TSPTW-D

### 5.1 Hiérarchie des problèmes

Le TSPTW-D s'inscrit dans une hiérarchie de problèmes dont le TSP est le cas de base. Chaque niveau ajoute une contrainte qui **restreint l'espace des solutions réalisables** tout en conservant (voire augmentant) la difficulté algorithmique.

| Problème | Contraintes actives | Espace réalisable $\mathcal{F}$ |
|---|---|---|
| **TSP** | Aucune (hormis la visite unique) | Toutes les permutations : $|\mathcal{F}| = (n-1)!$ |
| **TSPTW** | Fenêtres temporelles $[a_i, b_i]$ | $\mathcal{F}_{\text{TW}} \subseteq \mathcal{F}_{\text{TSP}}$ |
| **TSPTW-D** | Fenêtres temporelles + coûts dynamiques | $\mathcal{F}_{\text{D}} \subseteq \mathcal{F}_{\text{TW}}$ (mêmes permutations, coûts changés) |

**Relation d'inclusion :** toute solution réalisable de TSPTW-D est réalisable pour TSPTW (sous les coûts nominaux), qui est elle-même réalisable pour TSP. En revanche, l'inverse n'est pas vrai.

$$\mathcal{F}_{\text{TSPTW-D}} \subseteq \mathcal{F}_{\text{TSPTW}} \subseteq \mathcal{F}_{\text{TSP}}$$

### 5.2 Réductions formelles

**TSP → TSPTW :** poser $a_i = 0$, $b_i = +\infty$ et $s_i = 0$ pour tout $i$. Les fenêtres temporelles deviennent non contraignantes, et TSPTW se réduit au TSP.

**TSPTW → TSPTW-D :** poser $\delta_{ij}(t) = 0$ pour toute arête $\{i,j\}$ et tout $t \geq 0$. Les coûts deviennent constants, et TSPTW-D se réduit à TSPTW.

Ces deux réductions sont polynomiales et préservent la structure du problème — elles confirment que **TSPTW-D est au moins aussi difficile que TSP**.

### 5.3 Impact des contraintes sur l'espace de recherche

**Fenêtres temporelles :** elles élaguent l'espace de recherche en rendant infaisables les permutations qui ne respectent pas les plages horaires. Pour une instance donnée, le nombre de permutations réalisables dépend de la sévérité des fenêtres :
- Fenêtres larges → peu d'élagage, $\mathcal{F}_{\text{TW}} \approx \mathcal{F}_{\text{TSP}}$
- Fenêtres étroites → fort élagage, $|\mathcal{F}_{\text{TW}}| \ll (n-1)!$

**Coûts dynamiques :** ils ne changent pas l'ensemble des permutations réalisables (la faisabilité dépend des fenêtres, pas des coûts), mais ils modifient la **valeur de chaque solution** : une tournée optimale sous coûts statiques peut ne plus l'être sous coûts dynamiques, et inversement. Formellement :

$$\sigma^*_{\text{TSP}} = \arg\min_{\sigma \in \mathcal{F}_{\text{TSP}}} \sum_{k} c^{\text{base}}_{\{\sigma_k, \sigma_{k+1}\}}$$

$$\sigma^*_{\text{TSPTW-D}} = \arg\min_{\sigma \in \mathcal{F}_{\text{TW}}} \sum_{k} c_{\sigma_k, \sigma_{k+1}}(d_{\sigma_k})$$

Ces deux optima peuvent être très différents : une perturbation sur un arc emprunté en fin de tournée peut inciter à réordonner complètement les visites.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------
# Instance jouet : n=5 clients + dépôt (6 nœuds)
# On énumère TOUTES les permutations et on évalue leur faisabilité
# sous TSP, TSPTW et TSPTW-D pour illustrer la hiérarchie.
# ---------------------------------------------------------------

np.random.seed(0)
n = 5  # nombre de clients (dépôt = nœud 0)

# Coordonnées
coords = np.array([
    [0.5, 0.5],  # dépôt
    [0.1, 0.9],
    [0.9, 0.8],
    [0.8, 0.1],
    [0.2, 0.2],
    [0.6, 0.6],
])

# Coûts de base (distances euclidiennes × 100 → minutes)
def dist(i, j):
    return np.linalg.norm(coords[i] - coords[j]) * 100

cost_base = {(i, j): round(dist(i, j), 2)
             for i in range(n + 1) for j in range(n + 1) if i != j}

# Fenêtres temporelles et temps de service
time_windows = {
    0: (0, 500),
    1: (10,  80),
    2: (50, 150),
    3: (80, 200),
    4: (30, 120),
    5: (60, 180),
}
service_times = {i: 8 for i in range(n + 1)}

# Perturbation : arc (2,3) × 3 de t=60 à t=130
perturbations = [{"arc": (2, 3), "t_start": 60, "t_end": 130, "alpha": 3.0}]

def c_dynamic(i, j, t):
    base = cost_base[(i, j)]
    for p in perturbations:
        if p["arc"] == (i, j) and p["t_start"] <= t <= p["t_end"]:
            return base * p["alpha"]
    return base


def evaluate_tour(perm, dynamic=False):
    """
    Évalue une permutation (liste de clients 1..n).
    Retourne (faisable, coût_total) ou (False, inf).
    """
    route = [0] + list(perm) + [0]
    t = 0.0  # temps courant (départ du dépôt à t=0)
    total_cost = 0.0

    for k in range(len(route) - 1):
        i, j = route[k], route[k + 1]
        # Vérification fenêtre de départ
        ai, bi = time_windows[i]
        if t > bi:
            return False, float("inf")
        # Attente si arrivée avant ouverture
        t = max(t, ai) + service_times[i]
        # Transit
        travel = c_dynamic(i, j, t) if dynamic else cost_base[(i, j)]
        total_cost += travel
        t += travel

    # Vérification retour au dépôt dans la fenêtre
    a0, b0 = time_windows[0]
    if t > b0:
        return False, float("inf")
    return True, total_cost


# ---------------------------------------------------------------
# Énumération complète des (n-1)! / 2... ici on garde toutes les
# n! permutations pour compter faisabilité (n=5 → 120 permutations)
# ---------------------------------------------------------------
all_perms = list(itertools.permutations(range(1, n + 1)))

results = {"TSP": [], "TSPTW": [], "TSPTW-D": []}

for perm in all_perms:
    # TSP : toutes les permutations sont "faisables" (pas de contrainte de temps)
    cost_tsp = sum(cost_base[(([0] + list(perm) + [0])[k],
                              ([0] + list(perm) + [0])[k + 1])]
                   for k in range(n + 1))
    results["TSP"].append((True, cost_tsp))

    # TSPTW : coûts statiques + fenêtres temporelles
    ok_tw, cost_tw = evaluate_tour(perm, dynamic=False)
    results["TSPTW"].append((ok_tw, cost_tw))

    # TSPTW-D : coûts dynamiques + fenêtres temporelles
    ok_d, cost_d = evaluate_tour(perm, dynamic=True)
    results["TSPTW-D"].append((ok_d, cost_d))


def stats(res):
    feasible = [(ok, c) for ok, c in res if ok]
    n_feas = len(feasible)
    if n_feas == 0:
        return n_feas, float("inf"), float("inf")
    costs = [c for _, c in feasible]
    return n_feas, min(costs), np.mean(costs)


st_tsp  = stats(results["TSP"])
st_tw   = stats(results["TSPTW"])
st_d    = stats(results["TSPTW-D"])

print(f"{'Problème':<12} {'# réalisables':>14} {'Coût optimal':>14} {'Coût moyen':>12}")
print("-" * 56)
for name, st in [("TSP", st_tsp), ("TSPTW", st_tw), ("TSPTW-D", st_d)]:
    print(f"{name:<12} {st[0]:>14}  {st[1]:>13.1f}  {st[2]:>11.1f}")

# ---------------------------------------------------------------
# Visualisation : distributions des coûts réalisables
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
colors = {"TSP": "#2a9d8f", "TSPTW": "#e9c46a", "TSPTW-D": "#e76f51"}

opt_perm = {}
for name, res in results.items():
    feasible_costs = [c for ok, c in res if ok]
    ax = axes[list(results.keys()).index(name)]
    ax.hist(feasible_costs, bins=20, color=colors[name], edgecolor="white", alpha=0.9)
    opt = min(feasible_costs)
    opt_perm[name] = opt
    ax.axvline(opt, color="black", linewidth=2, linestyle="--", label=f"Optimum : {opt:.1f} min")
    ax.set_title(f"{name}\n({len(feasible_costs)}/{len(all_perms)} permutations réalisables)", fontsize=10)
    ax.set_xlabel("Coût de la tournée (minutes)", fontsize=9)
    ax.set_ylabel("Nombre de permutations", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Distribution des coûts de toutes les permutations (n=5 clients)\n"
             "Impact cumulatif des contraintes sur l'espace réalisable", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------
# Visualisation des tournées optimales pour chaque variante
# sur la même instance, pour montrer que les optima divergent.
# ---------------------------------------------------------------

def best_tour(results_dict, name, all_perms):
    res = results_dict[name]
    best_cost = float("inf")
    best_perm = None
    for perm, (ok, cost) in zip(all_perms, res):
        if ok and cost < best_cost:
            best_cost = cost
            best_perm = perm
    return best_perm, best_cost


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors_node = ["#f4a261"] + ["#457b9d"] * n
labels_nodes = {0: "Dépôt"}
labels_nodes.update({i: f"v{i}\n[{time_windows[i][0]},{time_windows[i][1]}]" for i in range(1, n + 1)})

pos = {i: tuple(coords[i]) for i in range(n + 1)}

for ax, name in zip(axes, ["TSP", "TSPTW", "TSPTW-D"]):
    best_perm, best_cost = best_tour(results, name, all_perms)
    route = [0] + list(best_perm) + [0]

    G = nx.Graph()
    G.add_nodes_from(range(n + 1))
    edges = [(route[k], route[k + 1]) for k in range(len(route) - 1)]
    G.add_edges_from(edges)

    # Arcs perturbés dans la tournée
    edge_colors = []
    for (i, j) in edges:
        if name == "TSPTW-D" and any(p["arc"] == (i, j) for p in perturbations):
            edge_colors.append("#e63946")
        else:
            edge_colors.append("#264653")

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors_node, node_size=700)
    nx.draw_networkx_labels(G, pos, labels=labels_nodes, ax=ax, font_size=6)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors, width=2.5,
                           )

    order_str = " → ".join([f"v{v}" for v in route])
    ax.set_title(f"{name}\nCoût optimal : {best_cost:.1f} min\n{order_str}", fontsize=9)
    ax.axis("off")

perturb_patch = mpatches.Patch(color="#e63946", label="Arc perturbé (×3)")
normal_patch  = mpatches.Patch(color="#264653", label="Arc normal")
fig.legend(handles=[normal_patch, perturb_patch], loc="lower center",
           ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle("Tournées optimales selon le modèle — même instance, résultats différents", fontsize=11)
plt.tight_layout()
plt.show()

print("\nConclusion :")
print(f"  TSP     → optimum à {best_tour(results, 'TSP',     all_perms)[1]:.1f} min "
      f"(toutes permutations autorisées)")
print(f"  TSPTW   → optimum à {best_tour(results, 'TSPTW',   all_perms)[1]:.1f} min "
      f"(fenêtres temporelles actives)")
print(f"  TSPTW-D → optimum à {best_tour(results, 'TSPTW-D', all_perms)[1]:.1f} min "
      f"(+ perturbation sur arc (2,3))")


---

## 6. Perspectives et extensions

La modélisation TSPTW-D présentée ici constitue une base solide pour les livrables suivants. Plusieurs extensions naturelles pourront être envisagées :

- **Perturbations stochastiques :** modéliser $\delta_{ij}(t)$ comme une variable aléatoire (loi log-normale, par exemple) pour traiter l'incertitude sur les temps de trajet. Cela conduit à des variantes de TSP robuste ou stochastique.

- **Violation souple des fenêtres temporelles :** dans certains contextes réels, une livraison légèrement en retard est préférable à une livraison annulée. On peut introduire un coût de pénalité $p_i(\tau_i)$ croissant avec le retard, et maximiser la qualité de service plutôt que d'imposer une contrainte dure.

- **Plusieurs véhicules (VRP) :** étendre le modèle à une flotte de véhicules, chacun effectuant sa propre sous-tournée depuis le dépôt. Le problème devient alors un *Vehicle Routing Problem with Time Windows* (VRPTW).

- **Recalcul en ligne :** en présence de perturbations imprévues détectées en cours de route, mettre en place un mécanisme de recalcul dynamique de la tournée (algorithme de réoptimisation).

---

## 7. Références

[1] Karp, R. M. (1972). *Reducibility among combinatorial problems*. In R. E. Miller & J. W. Thatcher (Eds.), *Complexity of Computer Computations* (pp. 85–103). Plenum Press.

[2] Savelsbergh, M. W. P. (1985). *Local search in routing problems with time windows*. *Annals of Operations Research*, 4(1), 285–305.

[3] Dumas, Y., Desrosiers, J., & Soumis, F. (1995). *The pickup and delivery problem with time windows*. *European Journal of Operational Research*, 54(1), 7–22.

[4] Gendreau, M., & Potvin, J.-Y. (2010). *Handbook of Metaheuristics* (2nd ed.). Springer.

[5] Ichoua, S., Gendreau, M., & Potvin, J.-Y. (2003). *Vehicle dispatching with time-dependent travel times*. *European Journal of Operational Research*, 144(2), 379–396.